# Recurrent-Llama-3.2 GSM8K 評価 (Colab T4 無料枠)

smcleish/Recurrent-Llama-3.2-train-recurrence-16 を `num_steps = 1, 2, 3, 4` で GSM8K 100問評価。

## 使い方
1. **ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ: T4 GPU** に設定
2. 上から順にセルを実行（Shift+Enter）
3. 最後のセルで結果 zip をダウンロード

想定実行時間: 約30〜60分（モデルロード5分 + 評価25〜55分）

In [ ]:
# === Step 1: 環境確認 ===
import torch, sys
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print('!!! GPU が割り当てられていません。ランタイム→タイプ変更で T4 GPU を選択してください !!!')

In [ ]:
# === Step 2: 依存ライブラリのインストール（論文時期に合わせた transformers 4.46.3）===
!pip install -q "transformers==4.46.3" "tokenizers<0.21" "huggingface_hub<0.30" "datasets>=3.0" "accelerate>=0.34" "matplotlib" "sentencepiece" "protobuf"

In [ ]:
# === Step 3: 評価スクリプト本体（eval_colab.py 相当をインライン化）===
%%writefile /content/eval_colab.py
import json, os, re, time
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
from tqdm.auto import tqdm

MODEL_ID = 'smcleish/Recurrent-Llama-3.2-train-recurrence-16'
N_SAMPLES = 100
NUM_STEPS_LIST = [1, 2, 3, 4]
MAX_NEW_TOKENS = 160
FEWSHOT_K = 4
OUTPUT_DIR = Path('/content/results')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

FEWSHOT_EXAMPLES = [
    ('There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?',
     'There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.'),
    ('If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?',
     'There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.'),
    ('Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?',
     'Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.'),
    ('Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?',
     'Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The answer is 8.'),
]

def build_prompt(q):
    parts = [f'Question: {x}\nAnswer: {y}' for x, y in FEWSHOT_EXAMPLES[:FEWSHOT_K]]
    parts.append(f'Question: {q}\nAnswer:')
    return '\n\n'.join(parts)

_NUM_RE = re.compile(r'-?\d[\d,]*(?:\.\d+)?')
def _norm(s): return s.replace(',', '').rstrip('.')
def extract_answer(text):
    cut = re.split(r'\n\s*Question:', text, maxsplit=1)[0]
    m = re.search(r'answer is\s*\$?(-?\d[\d,]*(?:\.\d+)?)', cut, re.IGNORECASE)
    if m: return _norm(m.group(1))
    nums = _NUM_RE.findall(cut)
    return _norm(nums[-1]) if nums else None
def gold_answer(s):
    m = re.search(r'####\s*(-?\d[\d,]*\.?\d*)', s)
    return m.group(1).replace(',', '') if m else s.strip()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'device={device}, dtype={dtype}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, trust_remote_code=True).to(device)
model.eval()

class StopOnSubstring(StoppingCriteria):
    def __init__(self, tok, stops, plen):
        self.tok, self.stops, self.plen = tok, stops, plen
    def __call__(self, input_ids, scores, **kw):
        gen = self.tok.decode(input_ids[0, self.plen:], skip_special_tokens=True)
        return any(s in gen for s in self.stops)

ds = load_dataset('gsm8k', 'main', split='test').select(range(N_SAMPLES))
all_results = {}
for num_steps in NUM_STEPS_LIST:
    print(f'\n=== num_steps = {num_steps} ===')
    n_correct, records = 0, []
    t0 = time.time()
    for i, ex in enumerate(tqdm(ds, desc=f'num_steps={num_steps}')):
        prompt = build_prompt(ex['question'])
        ids = tokenizer.encode(prompt, return_tensors='pt', add_special_tokens=True).to(device)
        stops = StoppingCriteriaList([StopOnSubstring(tokenizer, ['\nQuestion:', '\n\nQuestion:'], ids.shape[1])])
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 num_steps=num_steps, pad_token_id=tokenizer.eos_token_id,
                                 stopping_criteria=stops)
        gen = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        pred, gold = extract_answer(gen), gold_answer(ex['answer'])
        ok = pred is not None and pred == gold
        if ok: n_correct += 1
        records.append({'idx': i, 'question': ex['question'], 'gold': gold, 'pred': pred,
                        'gen_head': gen[:300], 'correct': ok})
    elapsed = time.time() - t0
    acc = n_correct / N_SAMPLES
    print(f'num_steps={num_steps}: accuracy = {acc:.3f} ({n_correct}/{N_SAMPLES}), time={elapsed:.1f}s')
    all_results[num_steps] = {'num_steps': num_steps, 'n_samples': N_SAMPLES,
                              'n_correct': n_correct, 'accuracy': acc,
                              'elapsed_sec': elapsed, 'records': records}
    with open(OUTPUT_DIR / f'gsm8k_num_steps_{num_steps}.json', 'w') as f:
        json.dump(all_results[num_steps], f, ensure_ascii=False, indent=2)

summary = {'model_id': MODEL_ID, 'n_samples': N_SAMPLES, 'device': device,
           'results': [{'num_steps': k, 'accuracy': v['accuracy'],
                        'n_correct': v['n_correct'], 'elapsed_sec': v['elapsed_sec']}
                       for k, v in all_results.items()]}
with open(OUTPUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('\nDONE. summary.json saved.')

In [ ]:
# === Step 4: 評価実行（25〜55分）===
%run /content/eval_colab.py

In [ ]:
# === Step 5: 結果サマリ表示 ===
import json
with open('/content/results/summary.json') as f:
    s = json.load(f)
print('Model:', s['model_id'])
print(f"\n{'num_steps':>10} | {'accuracy':>10} | {'time(s)':>10}")
print('-' * 38)
for r in s['results']:
    print(f"{r['num_steps']:>10} | {r['accuracy']:>10.3f} | {r['elapsed_sec']:>10.1f}")

In [ ]:
# === Step 6: 結果を zip でダウンロード ===
!cd /content && zip -r results_colab.zip results/
from google.colab import files
files.download('/content/results_colab.zip')